# VTuber Builder · Colab 기본 화면 (Gradio 없음)


## ① 환경 설치 · 실행 완료 후 셀 종료


In [ ]:
import os, pathlib, shutil, signal, subprocess, sys, time

REPO_URL = "https://github.com/jujumelona/Virtual-pipeline.git"
REPO_DIR = pathlib.Path("/content/Virtual-pipeline")
SETUP_LOG = pathlib.Path("/content/vtuber_builder/logs/colab_bootstrap.log")

def run(command, timeout):
    """Stream every stdout/stderr line in Colab and save the identical transcript."""
    SETUP_LOG.parent.mkdir(parents=True, exist_ok=True)
    label = subprocess.list2cmdline(command)
    print("[준비] $", label, "· 전체 로그:", SETUP_LOG, flush=True)
    with SETUP_LOG.open("a", encoding="utf-8") as output:
        output.write("\n$ " + label + "\n")
        output.flush()
        start_offset = output.tell()
        process = subprocess.Popen(
            command, stdout=output, stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    cursor = start_offset
    started = time.monotonic()

    def show_new_output():
        nonlocal cursor
        with SETUP_LOG.open("r", encoding="utf-8", errors="replace") as reader:
            reader.seek(cursor)
            while True:
                line = reader.readline()
                if not line:
                    break
                print(line, end="" if line.endswith("\n") else "\n", flush=True)
            cursor = reader.tell()

    try:
        while process.poll() is None:
            show_new_output()
            if time.monotonic() - started > timeout:
                raise TimeoutError(f"환경 준비 제한 시간 {timeout}초 초과 · {label}")
            time.sleep(0.1)
        show_new_output()
    except BaseException:
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=3)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
                process.wait(timeout=10)
        show_new_output()
        raise
    if process.returncode:
        raise RuntimeError(
            f"환경 준비 실패(exit={process.returncode}): {label}\n"
            f"모든 stdout/stderr가 위에 출력되었습니다. 파일: {SETUP_LOG}"
        )

print("① 저장소 동기화")
if (REPO_DIR / ".git").is_dir():
    run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], 60)
    run(["git", "-C", str(REPO_DIR), "fetch", "--prune", "origin", "main"], 180)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)], 300)
run(["git", "-C", str(REPO_DIR), "checkout", "-B", "main", "origin/main"], 60)
run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], 60)

sys.path.insert(0, str(REPO_DIR))
from tools.colab_native import stop_legacy_server
stop_legacy_server()

print("① Python / CUDA 환경 확인")
setup_code = (
    "import os,pathlib,runpy; os.environ['VTUBER_SETUP_ONLY']='1'; "
    "repo=pathlib.Path('/content/Virtual-pipeline'); "
    "app=runpy.run_path(str(repo/'tools'/'colab_app.py'),run_name='vtuber_prepare'); "
    "head,logs=app['ensure_runtime'](); "
    "print('준비 완료:',head[:12],flush=True)"
)
run([sys.executable, "-u", "-c", setup_code], 9000)
print("① 준비 완료. 이제 ②에서 모드를 선택하세요.")


## ② 생성 모드 및 용도 선택


In [ ]:
#@title ② 작업 종류 선택 · 캐릭터 생성이 기본값
#@markdown **먼저 '캐릭터 생성' 또는 '액세서리 제작'을 선택하세요.**
TASK = "캐릭터 생성" #@param ["캐릭터 생성", "액세서리 제작"]
#@markdown **캐릭터 생성인 경우에만 적용**
MODE = "3d" #@param ["3d", "inochi2d", "live2d"]
USAGE = "corporation" #@param ["corporation", "personalProfit", "personalNonProfit"]
EXISTING_IMAGE_PATH = "" #@param {type:"string"}
MULTI_REFERENCE_3D = False #@param {type:"boolean"}
#@markdown **아래 두 옵션은 액세서리 제작을 선택했을 때만 적용**
#@markdown AUTO: 파일명으로 위치 지정 · ALL: 각 이미지에 모든 지원 위치 적용
ACCESSORY_ANCHOR = "AUTO" #@param ["AUTO", "ALL", "HEAD_TOP", "FACE", "LEFT_EAR", "RIGHT_EAR", "NECK", "CHEST", "BACK", "LEFT_SHOULDER", "RIGHT_SHOULDER", "LEFT_HAND", "RIGHT_HAND", "LEFT_FOOT", "RIGHT_FOOT", "HIPS"]
ACCESSORY_BASE_VRM_PATH = "" #@param {type:"string"}
print(f"작업 종류: {TASK}")
if TASK == "캐릭터 생성":
    print(f"캐릭터 생성 방식: {MODE} · 사용 범위: {USAGE}")
else:
    print(f"액세서리 한 번에 생성 · 부착 범위: {ACCESSORY_ANCHOR}")
print("입력 이미지 경로가 비어 있으면 ③ 셀에서 Colab 파일 선택 창을 표시합니다.")


## ③ 캐릭터 사진 업로드 및 생성 (중단 가능)


In [ ]:
#@title ③ 선택한 작업 생성 · 실패 시 붉은 오류 표시
import pathlib, sys

repo = pathlib.Path("/content/Virtual-pipeline")
if not (repo / "tools" / "colab_native.py").is_file():
    raise RuntimeError("① 환경 설치 셀을 먼저 실행하세요.")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from tools.colab_native import generate

if TASK == "캐릭터 생성":
    if MODE not in ("3d", "inochi2d", "live2d"):
        raise ValueError(f"캐릭터 생성 방식이 잘못됐습니다: {MODE}")
    RESULT_FILE = generate(
        MODE, USAGE,
        image_path=EXISTING_IMAGE_PATH.strip(),
        full_body=MULTI_REFERENCE_3D,
    )
elif TASK == "액세서리 제작":
    RESULT_FILE = generate(
        "accessory", USAGE,
        accessory_anchor=ACCESSORY_ANCHOR,
        accessory_base_path=ACCESSORY_BASE_VRM_PATH.strip(),
    )
else:
    raise ValueError(f"작업 종류가 잘못됐습니다: {TASK}")

if not RESULT_FILE:
    # tools.colab_native.generate returns None for a failed callback. Without
    # this guard, Jupyter shows a green finished cell after the worker died.
    raise RuntimeError(
        "VTUBER_GENERATION_FAILED: 제작 결과가 없습니다. 위의 [stage failure] "
        "및 '결과:' 로그가 실제 실패 원인입니다. 이 셀은 성공하지 않았습니다. "
        "마지막 ④ 상태 진단 셀을 실행해 작업 로그를 확인하세요."
    )
RESULT_FILE = str(pathlib.Path(RESULT_FILE).resolve())
if not pathlib.Path(RESULT_FILE).is_file():
    raise RuntimeError("VTUBER_GENERATION_OUTPUT_MISSING: " + RESULT_FILE)
print("제작 성공 · 검증된 결과 파일:", RESULT_FILE, flush=True)
print("다운로드는 별도 셀에서: from google.colab import files; files.download(RESULT_FILE)")


결과 파일은 Colab 왼쪽 파일 탐색기에서 찾을 수 있습니다. 다운로드는 필요할 때만 별도 셀에서 `from google.colab import files; files.download(RESULT_FILE)`을 실행하세요.


## ④ 제작 상태 확인 · 런타임 재연결 후 진단


In [ ]:
#@title ④ 마지막 셀 · 실제 제작 상태 및 실패 로그 확인 (재연결 후에도 수동 실행 가능)
#@markdown 이 셀은 **모델 생성 완료를 자동으로 주장하지 않습니다.** 파일과 상태 기록을 검증합니다.
import json
from pathlib import Path

jobs = Path("/content/vtuber_builder/jobs")
reports = sorted(jobs.glob("*/status.json"), key=lambda x: x.stat().st_mtime, reverse=True) if jobs.is_dir() else []
result_file = globals().get("RESULT_FILE")
if reports:
    status_file = reports[0]
    data = json.loads(status_file.read_text(encoding="utf-8"))
    state = str(data.get("state", "unknown"))
    job_folder = status_file.parent
    print("최근 작업:", job_folder, flush=True)
    print("실제 작업 상태:", state, flush=True)
    print("worker PID:", data.get("pid"), "종료 코드:", data.get("exit_code", "기록 없음"), flush=True)
    print("전체 로그:", job_folder / "generation.log", flush=True)
    if state == "complete":
        if not result_file or not Path(str(result_file)).is_file():
            print("주의: 상태 기록은 complete이지만, 현재 세션의 결과 파일이 확인되지 않았습니다.", flush=True)
    elif state == "running":
        print("주의: running 기록만으로는 실제 프로세스가 살아 있다고 확인할 수 없습니다.", flush=True)
    else:
        print("제작 실패/취소 · 완성 모델로 취급하지 않습니다.", flush=True)
elif result_file and Path(str(result_file)).is_file():
    print("검증된 결과 파일:", Path(str(result_file)).resolve(), flush=True)
else:
    print("검증된 결과 파일이 없습니다. 작업 중 Colab 런타임이 삭제되면 /content 의 임시 로그도 사라질 수 있습니다.", flush=True)
print("Colab 오른쪽에 '런타임에 연결되어 있지 않습니다'가 표시된다면 이 셀도 실행할 수 없습니다.", flush=True)
print("먼저 Colab '다시 연결'을 시도해야 합니다. 무료 T4 할당량이 없다면 런타임이 재할당될 때까지 재실행할 수 없습니다.", flush=True)
